# 03 · Join Sofascore + Capology — Spain La Liga 23/24

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2023/24 de La Liga española**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [116]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [117]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [118]:
df_sf = pd.read_csv(SF_DIR / 'df_spain_2324.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_spain_2324.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  598 jugadores | 116 columnas
Capology:   556 jugadores | 9 columnas


## 4. Normalización

In [119]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [120]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   deportivo alaves
   girona fc

En Capology pero no en Sofascore:
   alaves
   girona


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [121]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'alaves':'deportivo alaves',
            'girona':'girona fc'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [122]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 489/598 (81.8%)
Sin emparejar: 109


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [123]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          8
Revisión media    (0.75 ≤ score < 0.90):   13
Revisión estricta (0.50 ≤ score < 0.75):   57
Revisión muy est. (score < 0.50):           31


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [124]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
4,Alexander Sørloth,Villarreal,alexander sorloth,0.970
3,Viktor Tsygankov,Girona FC,viktor tsyhankov,0.938
39,Javier Hernández,Cádiz,javi hernandez,0.933
33,Manuel Sánchez,Celta Vigo,manu sanchez,0.923
14,Daniel Vivian,Athletic Club,dani vivian,0.917
46,Javier Guerra,Valencia,javi guerra,0.917
35,Javier Muñoz,Las Palmas,javi munoz,0.909
12,Yéremy Pino,Villarreal,yeremi pino,0.909


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [125]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
38,Josep Chavarría,Rayo Vallecano,pep chavarria,0.857
56,Abdessamad Ezzalzouli,Real Betis,abde ezzalzouli,0.833
1,Savinho,Girona FC,savio,0.833
78,Ben Brereton Díaz,Villarreal,ben brereton,0.828
24,Abdelkabir Abqar,Deportivo Alavés,abdel abqar,0.815
23,José María Giménez,Atlético Madrid,jose gimenez,0.800
72,Iñaki González,Las Palmas,fabio gonzalez,0.786
15,José Luis Gayà,Valencia,jose gaya,0.783
29,Mamadou Mbaye,Cádiz,momo mbaye,0.783
54,Abderrahman Rebbach,Deportivo Alavés,abde rebbach,0.774


In [126]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['inaki gonzalez','damian rodriguez'
]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 11 | Excluidos: 2


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [127]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
57,Maximiliano Gómez,Cádiz,maxi gomez,0.741
20,Pathé Ismaël Ciss,Rayo Vallecano,pathe ciss,0.741
44,Raúl García de Haro,Osasuna,raul garcia,0.733
79,Urko González,Real Sociedad,urko gonzalez de zarate,0.722
51,Jorge Pascual,Villarreal,jorge cuenca,0.720
36,Jastin García,Girona FC,aleix garcia,0.720
2,Alejandro Baena,Villarreal,alex baena,0.720
70,Alberto Soro,Granada,alberto perea,0.720
52,Jesús Corona,Sevilla,jesus navas,0.696
40,Hannibal Mejbri,Sevilla,hannibal,0.696


In [128]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['maximiliano gomez',
                    'pathe ismael ciss',
                    'raul garcia de haro',
                    'urko gonzalez',
                    'alejandro baena',
                    'hannibal mejbri',
                    'anthony lozano',
                    'johnny cardoso',
                    'radamel falcao',
                    'pablo gavi',
                    'rodri sanchez',
                    'abner vinicius'
]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 12


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [129]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
30,Sokratis Papastathopoulos,Real Betis,sokratis,0.485
37,Mario Domínguez,Valencia,jaume domenech,0.483
74,Victor Parada,Deportivo Alavés,nikola maras,0.480
103,Gabri Veiga,Celta Vigo,oscar mingueza,0.480
73,Gorka Rivera,Getafe,borja mayoral,0.480
66,Yoel Lago,Celta Vigo,joseph aidoo,0.476
71,Selvi Clúa,Girona FC,eric garcia,0.476
84,Paco Sanz,Almería,choco lozano,0.476
76,Mamadou Sylla,Deportivo Alavés,aleksandar sedlar,0.467
88,Salvi Sánchez,Rayo Vallecano,aridane hernandez,0.467


In [130]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = ['sokratis papastathopoulos']
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 1


### 7.5 Aplicar todos los fuzzy matches aceptados

In [131]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 521/598 (87.1%)
Sin salario:     77


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [132]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 77


,player,team,minutesPlayed,appearances,goals,assists
0,Jonathan Viera,Almería,1542,21,2,4
1,Marcos Peña,Almería,439,7,0,0
2,Srđan Babić,Almería,90,1,0,0
3,Marciano Sanca Tchami,Almería,67,6,0,0
4,Paco Sanz,Almería,23,1,0,0
5,Rachad Fettal,Almería,16,1,0,0
6,Malcom Ares,Athletic Club,272,12,0,0
7,Mikel Jauregizar,Athletic Club,158,7,0,0
8,Aingeru Olabarrieta,Athletic Club,8,1,0,0
9,Salim El Jebari,Atlético Madrid,12,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [133]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Almería  —  SF sin salario:


,player,minutesPlayed
0,Jonathan Viera,1542
1,Marciano Sanca Tchami,67
2,Marcos Peña,439
3,Paco Sanz,23
4,Rachad Fettal,16
5,Srđan Babić,90


  CG plantilla completa:


,player,player_norm
0,Adrián Embarba,adrian embarba
1,Alejandro Pozo,alejandro pozo
2,Aleksandar Radovanović,aleksandar radovanovic
3,Álex Centelles,alex centelles
4,Arnau Puigmal,arnau puigmal
5,Bruno Langa,bruno langa
6,César Montes,cesar montes
7,Choco Lozano,choco lozano
8,Chumi,chumi
9,Diego Mariño,diego marino



  Athletic Club  —  SF sin salario:


...

,player,player_norm
0,Alberto Marí,alberto mari
1,André Almeida,andre almeida
2,Cenk Özkacar,cenk ozkacar
3,Cristhian Mosquera,cristhian mosquera
4,Cristian Rivero,cristian rivero
5,Diego López,diego lopez
6,Dimitri Foulquier,dimitri foulquier
7,Fran Pérez,fran perez
8,Gabriel Paulista,gabriel paulista
9,Giorgi Mamardashvili,giorgi mamardashvili



  Villarreal  —  SF sin salario:


,player,minutesPlayed
0,Jorge Pascual,13
1,Stefan Leković,28


  CG plantilla completa:


,player,player_norm
0,Adrià Altimira,adria altimira
1,Aïssa Mandi,aissa mandi
2,Alberto Moreno,alberto moreno
3,Álex Baena,alex baena
4,Alexander Sörloth,alexander sorloth
5,Alfonso Pedraza,alfonso pedraza
6,Arnaut Danjuma,arnaut danjuma
7,Ben Brereton,ben brereton
8,Bertrand Traoré,bertrand traore
9,Carlos Romero,carlos romero


In [134]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('jonathan viera', 'almeria') : ('jonathan viera', 'las palmas'),
    ('peter gonzalez', 'valencia'): ('peter federico', 'valencia'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')

Matches manuales definidos: 2


In [135]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: jonathan viera (almeria) → jonathan viera (las palmas)
✅ Match manual aplicado: peter gonzalez (valencia) → peter federico (valencia)

Tras matches manuales: 523/598 (87.5%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [136]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_spain_2324.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_spain_2324.csv
   Jugadores totales:  598
   Con salario:        523
   Sin salario (NaN):  75
   Columnas:           121
